In [1]:
import sys
import subprocess

# Instalar librerías si faltan
def install_if_missing(package, import_name=None):
    import_name = import_name or package
    try:
        __import__(import_name)
    except ImportError:
        print(f"Instalando {package}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", package, "-q"])

packages = [
    ("pandas", "pandas"),
    ("requests", "requests"),
    ("tqdm", "tqdm"),
]

for pkg, imp in packages:
    install_if_missing(pkg, imp)

print("✅ Dependencias listas")

✅ Dependencias listas


## Configuración de TMDB API

In [2]:
import os
import pandas as pd
import requests
import time
from tqdm import tqdm

# CONFIGURACIÓN DIRECTA
TMDB_API_KEY = "e5054f90fbd96ea53a646d27aff052e8"
TMDB_BASE_URL = "https://api.themoviedb.org/3"
TMDB_IMAGE_BASE_URL = "https://image.tmdb.org/t/p/w500"

# Rutas
DATA_RAW_PATH = 'data/raw/'
DATA_PROCESSED_PATH = 'data/processed/'
POSTERS_PATH = 'data/posters/'

# Crear directorios
os.makedirs(DATA_RAW_PATH, exist_ok=True)
os.makedirs(DATA_PROCESSED_PATH, exist_ok=True)
os.makedirs(POSTERS_PATH, exist_ok=True)

print("=" * 60)
print("✅ CONFIGURACIÓN LISTA")
print("=" * 60)
print(f"TMDB API Key: Configurada ✓")
print(f"Base URL: {TMDB_BASE_URL}")
print(f"Directorios creados ✓")

✅ CONFIGURACIÓN LISTA
TMDB API Key: Configurada ✓
Base URL: https://api.themoviedb.org/3
Directorios creados ✓


## Cliente TMDB

In [3]:
class TMDBClient:
    """Cliente para interactuar con API de TMDB"""
    
    def __init__(self, api_key):
        self.api_key = api_key
        self.base_url = TMDB_BASE_URL
        self.session = requests.Session()
        self.session.params = {
            'api_key': self.api_key,
            'language': 'es-ES'
        }
        self.last_request_time = 0
        self.request_delay = 0.2
    
    def _rate_limit(self):
        """Implementa rate limiting"""
        current_time = time.time()
        elapsed = current_time - self.last_request_time
        if elapsed < self.request_delay:
            time.sleep(self.request_delay - elapsed)
        self.last_request_time = time.time()
    
    def search_movie(self, title, year=None):
        """Busca una película por título"""
        self._rate_limit()
        
        url = f"{self.base_url}/search/movie"
        params = {'query': title}
        if year:
            params['year'] = year
        
        response = self.session.get(url, params=params)
        if response.status_code == 200:
            return response.json()
        return None
    
    def get_movie_details(self, movie_id):
        """Obtiene detalles de una película"""
        self._rate_limit()
        
        url = f"{self.base_url}/movie/{movie_id}"
        response = self.session.get(url)
        if response.status_code == 200:
            return response.json()
        return None
    
    def get_poster_url(self, poster_path, size="w500"):
        """Construye URL del póster"""
        if not poster_path:
            return None
        return f"{TMDB_IMAGE_BASE_URL}{poster_path}"

# Inicializar cliente
client = TMDBClient(TMDB_API_KEY)
print("✓ Cliente TMDB inicializado")

✓ Cliente TMDB inicializado


## Prueba: Buscar película en TMDB

In [4]:
print("=" * 60)
print("🔍 PROBANDO BÚSQUEDA EN TMDB")
print("=" * 60)

result = client.search_movie("The Matrix")
if result and result.get('results'):
    movie = result['results'][0]
    print(f"\n✅ Película encontrada:")
    print(f"   Título: {movie.get('title')}")
    print(f"   ID TMDB: {movie.get('id')}")
    print(f"   Año: {movie.get('release_date', 'N/A')[:4]}")
    print(f"   Rating: {movie.get('vote_average', 0)}/10")
    print(f"   Descripción: {movie.get('overview', 'N/A')[:100]}...")
    
    # Obtener detalles completos
    details = client.get_movie_details(movie['id'])
    if details:
        print(f"\n✅ Detalles adicionales:")
        print(f"   Género: {[g['name'] for g in details.get('genres', [])]}")
        print(f"   Duración: {details.get('runtime')} min")
        print(f"   Presupuesto: ${details.get('budget', 0):,}")
else:
    print("❌ No se encontraron resultados")

🔍 PROBANDO BÚSQUEDA EN TMDB

✅ Película encontrada:
   Título: Matrix
   ID TMDB: 603
   Año: 1999
   Rating: 8.247/10
   Descripción: Thomas Anderson lleva una doble vida: por el día es programador en una importante empresa de softwar...

✅ Detalles adicionales:
   Género: ['Acción', 'Ciencia ficción']
   Duración: 138 min
   Presupuesto: $63,000,000


## Funciones auxiliares: Descargar Imágenes

In [5]:
def download_image(url, save_path):
    """Descarga una imagen desde una URL"""
    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()
        with open(save_path, 'wb') as f:
            f.write(response.content)
        return True
    except Exception as e:
        print(f"Error descargando {url}: {e}")
        return False

def get_poster_filename(film_id, title):
    """Genera nombre seguro para archivo de póster"""
    safe_title = title.replace('/', '_').replace('\\', '_')[:30]
    return f"{film_id:04d}_{safe_title}.jpg"

print("✓ Funciones de descarga listas")

✓ Funciones de descarga listas


## Descarga de Pósters

In [6]:
# Lista de películas para descargar pósters
movies_to_download = [
    ("The Matrix", 1999),
    ("Avatar", 2009),
    ("Inception", 2010),
    ("Interstellar", 2014),
    ("Titanic", 1997),
]

print("=" * 60)
print("📥 DESCARGA DE PÓSTERS")
print("=" * 60)
print(f"\nDescargando pósters de {len(movies_to_download)} películas...\n")

downloaded_posters = []

for title, year in tqdm(movies_to_download, desc="Pósters"):
    result = client.search_movie(title, year)
    
    if result and result.get('results'):
        movie = result['results'][0]
        movie_id = movie.get('id')
        poster_path = movie.get('poster_path')
        
        log_entry = {
            'film_id': movie_id,
            'title': movie.get('title'),
            'poster_path': poster_path,
            'poster_downloaded': False,
            'poster_file': None,
            'error': None
        }
        
        if poster_path:
            poster_url = client.get_poster_url(poster_path)
            filename = get_poster_filename(movie_id, movie.get('title'))
            filepath = os.path.join(POSTERS_PATH, filename)
            
            if download_image(poster_url, filepath):
                log_entry['poster_downloaded'] = True
                log_entry['poster_file'] = filename
            else:
                log_entry['error'] = 'Download failed'
        else:
            log_entry['error'] = 'No poster path'
        
        downloaded_posters.append(log_entry)
    else:
        print(f"⚠️  No encontrada: {title}")

# Guardar log
if downloaded_posters:
    df_log = pd.DataFrame(downloaded_posters)
    log_path = os.path.join(DATA_PROCESSED_PATH, 'poster_download_log.csv')
    df_log.to_csv(log_path, index=False)
    
    print(f"\n✅ Pósters descargados: {df_log['poster_downloaded'].sum()}/{len(df_log)}")
    print(f"   Log guardado en: {log_path}")
    print(f"\n📊 Resultado:")
    print(df_log.to_string())

📥 DESCARGA DE PÓSTERS

Descargando pósters de 5 películas...



Pósters: 100%|██████████| 5/5 [00:04<00:00,  1.11it/s]


✅ Pósters descargados: 5/5
   Log guardado en: data/processed/poster_download_log.csv

📊 Resultado:
   film_id         title                       poster_path  poster_downloaded              poster_file error
0      603        Matrix  /8rT9kG2EYkZpJmYCuTJNnPDEube.jpg               True          0603_Matrix.jpg  None
1    19995        Avatar  /t5T3LPbLLgP2OP6kloM9p2PXpJL.jpg               True         19995_Avatar.jpg  None
2    27205        Origen   /tXQvtRWfkUUnWJAn2tN3jERIUG.jpg               True         27205_Origen.jpg  None
3   157336  Interstellar  /9cTfZWP5TfdnmAjiD6ZBXWIJ7O9.jpg               True  157336_Interstellar.jpg  None
4      597       Titanic  /rBTJZrf5UWaxzg5YJd2eqpeaSvm.jpg               True         0597_Titanic.jpg  None


## Función: Buscar película en TMDB

In [7]:
def search_film_in_tmdb(client, film_title, release_year=None):
    """Busca una película en TMDB y retorna sus detalles"""
    result = client.search_movie(film_title, release_year)
    if result and result.get('results'):
        return result['results'][0]
    return None

# Ejemplo de uso
print("=" * 60)
print("🎬 BÚSQUEDA AVANZADA EN TMDB")
print("=" * 60)

test_movie = search_film_in_tmdb(client, "The Dark Knight", 2008)
if test_movie:
    print(f"\n✅ Película encontrada: {test_movie.get('title')}")
    print(f"   ID: {test_movie.get('id')}")
    print(f"   Release: {test_movie.get('release_date')}")
    print(f"   Rating: {test_movie.get('vote_average')}/10")
    print(f"   Popularidad: {test_movie.get('popularity')}")
else:
    print("❌ No se encontró la película")

🎬 BÚSQUEDA AVANZADA EN TMDB

✅ Película encontrada: El caballero oscuro
   ID: 155
   Release: 2008-07-16
   Rating: 8.531/10
   Popularidad: 45.1293


## Descarga por lotes de películas

In [8]:
def download_movies_batch(client, num_movies=600):
    """
    Descarga metadata de películas populares de TMDB
    
    Args:
        client: Instancia de TMDBClient
        num_movies: Número de películas a descargar (~600)
    
    Returns:
        DataFrame con películas descargadas
    """
    all_movies = []
    page = 1
    
    while len(all_movies) < num_movies:
        try:
            client._rate_limit()
            url = f"{client.base_url}/movie/popular"
            params = {
                'api_key': TMDB_API_KEY,
                'language': 'es-ES',
                'page': page
            }
            
            response = requests.get(url, params=params)
            
            if response.status_code == 200:
                data = response.json()
                movies = data.get('results', [])
                
                if not movies:
                    break
                
                for movie in movies:
                    if len(all_movies) >= num_movies:
                        break
                    
                    all_movies.append({
                        'film_id': movie.get('id'),
                        'title': movie.get('title'),
                        'release_date': movie.get('release_date'),
                        'vote_average': movie.get('vote_average'),
                        'overview': movie.get('overview'),
                        'popularity': movie.get('popularity')
                    })
                
                page += 1
            else:
                break
        except:
            break
    
    return pd.DataFrame(all_movies)


def download_reviews_from_movies(client, movies_df, target_reviews=1500):
    """
    Descarga reseñas de películas ya descargadas
    
    Args:
        client: Instancia de TMDBClient
        movies_df: DataFrame con película (necesita columna 'film_id')
        target_reviews: Número total de reseñas a descargar (~1500)
    
    Returns:
        DataFrame con todas las reseñas descargadas
    """
    all_reviews = []
    downloaded = 0
    print(f"🎯 OBJETIVO: Descargar {target_reviews} reseñas")
    
    for _, row in tqdm(movies_df.iterrows(), total=len(movies_df), desc="Descargando reseñas"):
        if downloaded >= target_reviews:
            break
        
        movie_id = row['film_id']
        movie_title = row['title']
        
        # Descargar reseñas hasta 20 páginas por película
        for page in range(1, 21):
            if downloaded >= target_reviews:
                break
            
            try:
                client._rate_limit()
                url = f"{client.base_url}/movie/{movie_id}/reviews"
                params = {
                    'api_key': TMDB_API_KEY,
                    'language': 'es-ES',
                    'page': page
                }
                
                response = requests.get(url, params=params)
                
                if response.status_code == 200:
                    data = response.json()
                    reviews = data.get('results', [])
                    
                    if not reviews:
                        break
                    
                    for review in reviews:
                        if downloaded >= target_reviews:
                            break
                        
                        content = review.get('content', '')
                        if len(content) >= 50:
                            all_reviews.append({
                                'movie_id': movie_id,
                                'title': movie_title,
                                'author': review.get('author', 'Anonymous'),
                                'content': content,
                                'rating': review.get('author_details', {}).get('rating'),
                                'created_at': review.get('created_at')
                            })
                            downloaded += 1
                            # Mostrar progreso cada 100 reseñas
                            if downloaded % 100 == 0:
                                print(f"   ⏳ Descargadas {downloaded}/{target_reviews} reseñas...")
            except:
                continue
    
    return pd.DataFrame(all_reviews)

# PASO 1: Descargar 600 películas populares
print("=" * 60)
print("📥 PASO 1: DESCARGANDO 600 PELÍCULAS")
print("=" * 60)

df_movies = download_movies_batch(client, num_movies=600)

# Guardar películas
movies_csv_path = os.path.join(DATA_PROCESSED_PATH, 'movies_batch_1500.csv')
df_movies.to_csv(movies_csv_path, index=False)

print(f"\n✅ Películas descargadas: {len(df_movies)}")
print(f"   Guardadas en: {movies_csv_path}")
print(f"\n📊 Primeras películas:")
print(df_movies[['title', 'release_date', 'vote_average']].head())

# PASO 2: Descargar 1500 reseñas de esas películas
print("\n" + "=" * 60)
print("📥 PASO 2: DESCARGANDO 1500 RESEÑAS")
print("=" * 60)

df_reviews = download_reviews_from_movies(client, df_movies, target_reviews=1500)

# Guardar reseñas
reviews_csv_path = os.path.join(DATA_PROCESSED_PATH, 'reviews_batch_1500.csv')
df_reviews.to_csv(reviews_csv_path, index=False)

print(f"\n✅ Reseñas descargadas: {len(df_reviews)}")
print(f"   Películas con reseñas: {df_reviews['movie_id'].nunique()}")
print(f"   Guardadas en: {reviews_csv_path}")
print(f"\n📊 Muestra de reseñas:")
print(df_reviews[['title', 'author', 'rating']].head())

📥 PASO 1: DESCARGANDO 600 PELÍCULAS

✅ Películas descargadas: 600
   Guardadas en: data/processed/movies_batch_1500.csv

📊 Primeras películas:
                                        title release_date  vote_average
0                                    Obsesión   2026-05-13         7.907
1                      La momia de Lee Cronin   2026-04-15         8.065
2                                        Kara   2026-04-30         5.600
3  Jack Ryan de Tom Clancy: Guerra Encubierta   2026-05-20         7.134
4                 The Punisher: One Last Kill   2026-05-12         8.375

📥 PASO 2: DESCARGANDO 1500 RESEÑAS
🎯 OBJETIVO: Descargar 1500 reseñas


Descargando reseñas:  45%|████▌     | 271/600 [03:15<04:09,  1.32it/s]

   ⏳ Descargadas 100/1500 reseñas...


Descargando reseñas:  93%|█████████▎| 557/600 [06:47<00:41,  1.05it/s]

   ⏳ Descargadas 200/1500 reseñas...


Descargando reseñas: 100%|██████████| 600/600 [07:21<00:00,  1.36it/s]


✅ Reseñas descargadas: 221
   Películas con reseñas: 146
   Guardadas en: data/processed/reviews_batch_1500.csv

📊 Muestra de reseñas:
                      title                      author  rating
0        Proyecto Salvación                     anamb25     6.0
1                    Matrix  Antonio Alaminos-Fernández     9.0
2    Avatar: Fuego y ceniza                      Cuby75     6.0
3    Avatar: Fuego y ceniza                  by.annie._     NaN
4  El diablo viste de Prada                   Jrloridan     9.0


## Resumen Final

In [9]:
print("=" * 60)
print("📊 RESUMEN FINAL")
print("=" * 60)

# Verificar archivos generados
import glob

poster_files = glob.glob(os.path.join(POSTERS_PATH, "*.jpg"))
csv_files = glob.glob(os.path.join(DATA_PROCESSED_PATH, "*.csv"))

print(f"\n📁 Archivos generados:")
print(f"   Pósters descargados: {len(poster_files)}")
print(f"   Archivos CSV: {len(csv_files)}")

print(f"\n📂 Ubicación de archivos:")
print(f"   Pósters: {os.path.abspath(POSTERS_PATH)}")
print(f"   Datos procesados: {os.path.abspath(DATA_PROCESSED_PATH)}")

print(f"\n✅ PIPELINE COMPLETADO")
print(f"   - Cliente TMDB funcionando ✓")
print(f"   - Búsqueda de películas ✓")
print(f"   - Descarga de pósters ✓")
print(f"   - Generación de logs ✓")

📊 RESUMEN FINAL

📁 Archivos generados:
   Pósters descargados: 5
   Archivos CSV: 3

📂 Ubicación de archivos:
   Pósters: c:\Users\ASUS\Desktop\SIS-INTE\data\posters
   Datos procesados: c:\Users\ASUS\Desktop\SIS-INTE\data\processed

✅ PIPELINE COMPLETADO
   - Cliente TMDB funcionando ✓
   - Búsqueda de películas ✓
   - Descarga de pósters ✓
   - Generación de logs ✓
